In [2]:
import pandas as pd
df = pd.read_excel('Entregable 2 - Documentación extra.xlsx')

In [2]:
df.head()

,",country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery",Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,"0,Italy,""Aromas include tropical fruit, broom,...",NaN,NaN,NaN,NaN
1,"1,Portugal,""This is ripe and fruity, a wine th...",NaN,NaN,NaN,NaN
2,"2,US,""Tart and snappy, the flavors of lime fle...",NaN,NaN,NaN,NaN
3,"3,US,""Pineapple rind, lemon pith and orange bl...",NaN,NaN,NaN,NaN
4,"4,US,""Much like the regular bottling from 2012...",NaN,NaN,NaN,NaN


Aparentemente el archivo es un excel pero mal configurado. Parecería ser que todo el contenido está en una sola columna con sus valores separados por coma. Voy a revisar si las otras 3 columnas (unnamed) tienen algun dato.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 129975 entries, 0 to 129974
Data columns (total 5 columns):
 #   Column                                                                                                                           Non-Null Count   Dtype
---  ------                                                                                                                           --------------   -----
 0   ,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery  129975 non-null  str  
 1   Unnamed: 1                                                                                                                       2956 non-null    str  
 2   Unnamed: 2                                                                                                                       159 non-null     str  
 3   Unnamed: 3                                                                                                     

Efectivamente, todos los Unnamed tienen datos. Y esa cantidad de datos o filas no nulas parece ir en cascada, es decir del Unnamed 1 al 5 la cantidad de filas no nulas va decreciendo, siendo 2956 en el 1 y 1 en el 5

In [20]:
mascara = df['Unnamed: 1'].notna()
primer_notna = df.loc[mascara].iloc[0]

In [21]:
for x in primer_notna:
    print(x)

48,US,"This bottling resembles the New Zealand paradigm of Sauvignon Blanc, bearing aromas of grapefruit, passion fruit and kiwi
 a sprinkling of graham cracker adds interest. The wine hits the palate like a fleshy fist, with an intense, grassy gooseberry flavor that provides plenty of punch. Pair with apricot-glazed roasted chicken.",,86,16.0,Virginia,Monticello,,,,Trump 2011 Sauvignon Blanc (Monticello),Sauvignon Blanc,Trump
nan
nan
nan


Parece que el motivo de las columnas unnamed es que el texto se parte en dos por una coma mal parseada como delimitador, partiendo asi el texto. 

In [3]:
len(df.columns[0].split(','))

14

Hay 14 campos, lo que me indica que toda línea reconstruida, parseada como CSV, debe producir exactamente 14 campos. Sin embargo, uno de ellos es el índice.   

In [6]:
import re

col = df[df.columns[0]]                      # la columna larga, sin tipear el nombre monstruoso
for ch in [',', '.', ';', ':', '|', '\t']:
    print(repr(ch), col.str.count(re.escape(ch)).sum())

',' 2096660
'.' 490443
';' 0
':' 1626
'|' 1
'\t' 0


Parece que el delimitador que rompio el archivo fue el ; sabiendo esto vamos a reconstruirlo.

In [7]:
import io, re, csv
from collections import Counter

RUTA = 'Entregable 2 - Documentación extra.xlsx'

# 'crudo' es el Excel tal cual: 5 columnas inútiles donde debería haber 13.
crudo = pd.read_excel(RUTA)

# read_excel interpretó la primera línea del CSV como encabezado y la convirtió
# en el NOMBRE de la columna 0. Hay que devolverla a la lista de líneas o
# perdemos los nombres de las columnas.
lineas = [crudo.columns[0]]

# Excel partió cada línea en el ';' que usó como delimitador y lo descartó.
# Para deshacerlo: pegar las celdas no nulas de cada fila con ';' en el medio.
# El isinstance(c, str) descarta los NaN (las celdas vacías no son texto).
for fila in crudo.itertuples(index=False):
    lineas.append(';'.join(c for c in fila if isinstance(c, str)))

print(len(lineas), 'líneas reconstruidas')

129976 líneas reconstruidas


In [8]:
lineas_ok = [lineas[0]]
for l in lineas[1:]:
    if re.match(r'^\d+,', l):
        lineas_ok.append(l)
    else:
        lineas_ok[-1] += '\n' + l

texto = '\n'.join(lineas_ok)
print(len(lineas), '->', len(lineas_ok), 'líneas tras reunir las partidas')

129976 -> 129972 líneas tras reunir las partidas


In [9]:
# Criterio de éxito definido de antemano: si la reconstrucción es correcta,
# TODA línea parseada como CSV debe dar exactamente 14 campos
# (13 columnas de datos + la columna de índice sin nombre).
# csv.reader respeta el entrecomillado; un .split(',') acá daría cualquier cosa.
print(Counter(len(f) for f in csv.reader(io.StringIO(texto))))

Counter({14: 129972})


In [10]:
# index_col=0 usa esa primera columna sin nombre como índice del DataFrame,
# en vez de arrastrarla como una columna de datos redundante.
df = pd.read_csv(io.StringIO(texto), index_col=0)

assert df.shape[1] == 13, f'esperaba 13 columnas, hay {df.shape[1]}'
assert df['points'].dtype.kind == 'i', 'points debería ser entero'

print(df.shape)
df.info()

(129971, 13)
<class 'pandas.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   country                129908 non-null  str    
 1   description            129971 non-null  str    
 2   designation            92506 non-null   str    
 3   points                 129971 non-null  int64  
 4   price                  120975 non-null  float64
 5   province               129908 non-null  str    
 6   region_1               108724 non-null  str    
 7   region_2               50511 non-null   str    
 8   taster_name            103727 non-null  str    
 9   taster_twitter_handle  98758 non-null   str    
 10  title                  129971 non-null  str    
 11  variety                129970 non-null  str    
 12  winery                 129971 non-null  str    
dtypes: float64(1), int64(1), str(11)
memory usage: 12.9 MB


In [11]:
from analisis_exploratorio_vinos import cargar_datos

df = cargar_datos()
df.head(3)

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm


In [12]:
perfil = pd.DataFrame({
    'tipo':      df.dtypes.astype(str),
    'no_nulos':  df.notna().sum(),
    'nulos':     df.isna().sum(),
    'nulos_%':   (df.isna().mean() * 100).round(2),
    'unicos':    df.nunique(),
    'cardinalidad_%': (df.nunique() / len(df) * 100).round(2),
})
perfil.sort_values('nulos_%', ascending=False)


,tipo,no_nulos,nulos,nulos_%,unicos,cardinalidad_%
region_2,str,50511,79460,61.14,17,0.01
designation,str,92506,37465,28.83,37979,29.22
taster_twitter_handle,str,98758,31213,24.02,15,0.01
taster_name,str,103727,26244,20.19,19,0.01
region_1,str,108724,21247,16.35,1229,0.95
price,float64,120975,8996,6.92,390,0.30
province,str,129908,63,0.05,425,0.33
country,str,129908,63,0.05,43,0.03
description,str,129971,0,0.00,119955,92.29
points,int64,129971,0,0.00,21,0.02


In [13]:
print('duplicados exactos (13 columnas):', df.duplicated().sum())
print('misma reseña (description + title):', df.duplicated(subset=['description', 'title']).sum())

duplicados exactos (13 columnas): 9983
misma reseña (description + title): 9983


In [17]:
num = df[['points', 'price']]
resumen = num.describe().T
resumen['mediana']  = num.median()
resumen['IQR']      = num.quantile(.75) - num.quantile(.25)
resumen['asimetria'] = num.skew()      # 0 = simétrica; >1 = cola derecha larga
resumen['curtosis']  = num.kurtosis()  # >3 = colas más pesadas que una normal
resumen['CV']        = num.std() / num.mean()   # dispersión relativa, comparable entre variables
resumen.round(2)

,count,mean,std,min,25%,50%,75%,max,mediana,IQR,asimetria,curtosis,CV
points,129971.0,88.45,3.04,80.0,86.0,88.0,91.0,100.0,88.0,5.0,0.05,-0.30,0.03
price,120975.0,35.36,41.02,4.0,17.0,25.0,42.0,3300.0,25.0,25.0,18.00,829.52,1.16


In [18]:
for col in ['country', 'variety', 'taster_name', 'province', 'winery']:
    vc = df[col].value_counts()
    top10 = vc.head(10).sum() / vc.sum() * 100
    print(f'--- {col}: {df[col].nunique()} categorías | top-10 concentra {top10:.1f}%')
    print(vc.head(5).to_string(), '\n')

--- country: 43 categorías | top-10 concentra 95.9%
country
US          54504
France      22093
Italy       19540
Spain        6645
Portugal     5691 

--- variety: 707 categorías | top-10 concentra 54.9%
variety
Pinot Noir                  13272
Chardonnay                  11753
Cabernet Sauvignon           9472
Red Blend                    8946
Bordeaux-style Red Blend     6915 

--- taster_name: 19 categorías | top-10 concentra 92.1%
taster_name
Roger Voss           25514
Michael Schachner    15134
Kerin O’Keefe        10776
Virginie Boone        9537
Paul Gregutt          9532 

--- province: 425 categorías | top-10 concentra 61.3%
province
California    36247
Washington     8639
Bordeaux       5941
Tuscany        5897
Oregon         5373 

--- winery: 16757 categorías | top-10 concentra 1.5%
winery
Wines & Winemakers    222
Testarossa            218
DFJ Vinhos            215
Williams Selyem       211
Louis Latour          199 

